# LangChain Integration with Langfuse

This notebook mirrors `udemy-langfuse/instrumentation_langchain.py`. It uses the Langfuse LangChain `CallbackHandler` to trace a simple LCEL chain.

Before running it, configure `ANTHROPIC_API_KEY`, `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, and `LANGFUSE_BASE_URL`.

## 1. Define the Chain

The callback handler traces LangChain prompt formatting and model calls. The outer `@observe` wrapper gives the whole run a named parent observation.

In [1]:
from inspect import signature
from typing import Any

from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langfuse import Langfuse, get_client, observe, propagate_attributes
from langfuse.langchain import CallbackHandler

load_dotenv()

# Langfuse reads LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY, and LANGFUSE_BASE_URL.
langfuse = get_client()

In [2]:
def create_langfuse_handler(trace_seed: str) -> CallbackHandler:
    """Create a Langfuse LangChain callback handler.

    Newer Langfuse SDK versions accept `langfuse_client=...`; some installed
    versions accept only `trace_context=...`. This keeps the example compatible
    while still using the current callback-based integration pattern.
    """
    trace_context = {"trace_id": Langfuse.create_trace_id(seed=trace_seed)}
    handler_params = signature(CallbackHandler).parameters

    if "langfuse_client" in handler_params:
        return CallbackHandler(
            langfuse_client=langfuse,
            trace_context=trace_context,
        )

    return CallbackHandler(trace_context=trace_context)


@observe(name="run_langchain_example", as_type="chain")
def run_langchain_example(
    topic: str = "quantum computing",
    user_id: str = "demo-user",
    session_id: str | None = "langchain-demo-session",
) -> str:
    """Run a LangChain LCEL chain and trace it with Langfuse."""
    handler = create_langfuse_handler(trace_seed=f"langchain-{topic}")

    # Trace-level attributes are propagated to the observed wrapper and child
    # LangChain callback observations where supported by the SDK.
    with propagate_attributes(
        user_id=user_id,
        session_id=session_id,
        tags=["langchain", "callback-handler", "demo"],
        metadata={"topic": topic, "framework": "langchain"},
        trace_name="langchain-demo",
    ):
        # ChatAnthropic is the LangChain chat-model wrapper for Anthropic.
        llm = ChatAnthropic(model="claude-sonnet-4-20250514")

        # The prompt variable name must match the dict passed to chain.invoke().
        prompt = ChatPromptTemplate.from_template(
            "Explain {topic} in simple terms."
        )
        chain = prompt | llm

        # Attach the Langfuse callback at invocation time. The callback captures
        # the prompt step, model generation, token usage, latency, metadata, and tags.
        response = chain.invoke(
            {"topic": topic},
            config={
                "callbacks": [handler],
                "metadata": {"use_case": "langchain_example"},
                "tags": ["course", "langchain"],
            },
        )

    langfuse.update_current_span(
        output={"content": response.content},
        metadata={"topic": topic},
    )
    return response.content


## 2. Run the Example

Run the chain, then inspect Langfuse for the parent chain, prompt step, generation, latency, token usage, metadata, and tags.

In [3]:
answer = run_langchain_example(
    topic="quantum computing",
    user_id="course-user",
    session_id="langchain-demo-session",
)
print(answer)

# Flush after notebook cells so observations appear immediately.
langfuse.flush()


**Quantum computing is like having a super-powered computer that works in a fundamentally different way.**

## Regular computers vs. Quantum computers

**Regular computers** use bits that are either 0 or 1 - like light switches that are either OFF or ON. They process information one step at a time.

**Quantum computers** use "qubits" that can be 0, 1, or *both at the same time* - imagine a coin that's spinning in the air before it lands. This weird "both states at once" property is called **superposition**.

## Why this matters

Because qubits can exist in multiple states simultaneously, quantum computers can:
- Explore many possible solutions to a problem at once
- Solve certain types of problems exponentially faster than regular computers

Think of it like finding your way through a maze: a regular computer tries one path at a time, while a quantum computer can explore all paths simultaneously.

## What they're good at

Quantum computers excel at specific tasks like:
- Breaking encry